# 00 — Kalshi Market Selection

Pull the full Kalshi market catalog, classify by crypto relevance tier, apply liquidity filters,
and produce a final list of markets to use in the analysis.

**Output**: `data/raw/kalshi/market_catalog.parquet` + printed final market list

In [1]:
import sys
sys.path.append('..')

import os
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from src.fetch import (
    fetch_all_kalshi_markets,
    fetch_kalshi_historical_markets,
    fetch_kalshi_candlesticks,
    save_kalshi_market,
    KALSHI_BASE,
)

# Load API keys from .env in the project root
load_dotenv('../.env')

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

API_KEY = os.environ.get('KALSHI_API_KEY', None)
print('API key set:', API_KEY is not None)

API key set: True


/Users/kianjagtiani/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Pull Crypto-Relevant Markets

Rather than fetching all 600k+ Kalshi markets, we query only known crypto-relevant series tickers directly. This keeps the catalog to a manageable size (expected: ~50–500 markets).

In [2]:
import requests

# KXBTC and KXETH are Kalshi's daily price-bracket markets (200k+ each) — NOT what we want.
# We only want long-duration EVENT markets. Use specific series tickers instead.

CRYPTO_SERIES = [
    # Long-horizon BTC/ETH event markets
    'KXBTCMAXY',       # "How high will BTC get this year?"
    'KXBTCMAX100',     # "When will BTC hit $100K?"
    'KXBTCMAX125',     # "When will BTC hit $125K?"
    'KXBTCMAX150',     # "When will BTC hit $150K?"
    'KXBTC2025100',    # "Will BTC cross $100K again this year?"
    'KXBTCRESERVE',    # Bitcoin strategic reserve
    'KXUSDCMIN',       # USDC de-peg
    # Regulatory / ETF
    'KXCRYPTO',        # General crypto regulation
    'KXSOLANAETF',     # Solana ETF approval
    # Macro (affect crypto)
    'KXFED',           # FOMC rate decisions
    'KXCPI',           # CPI / inflation
    'KXGDP',           # GDP growth
    'KXUNEMPLOYMENT',  # Unemployment
]

def fetch_markets_by_series(series_ticker, api_key=None):
    headers = {'Authorization': f'Bearer {api_key}'} if api_key else {}
    markets = []
    cursor = None
    while True:
        params = {'series_ticker': series_ticker, 'limit': 1000}
        if cursor:
            params['cursor'] = cursor
        resp = requests.get(f'{KALSHI_BASE}/markets', params=params, headers=headers)
        if resp.status_code != 200:
            print(f'    HTTP {resp.status_code} for {series_ticker}')
            break
        data = resp.json()
        batch = data.get('markets', [])
        markets.extend(batch)
        cursor = data.get('cursor')
        if not cursor or not batch:
            break
    return markets

print('Fetching event markets from targeted series...')
all_markets_list = []

for series in CRYPTO_SERIES:
    batch = fetch_markets_by_series(series, api_key=API_KEY)
    print(f'  {series}: {len(batch)} markets')
    all_markets_list.extend(batch)

all_markets = pd.DataFrame(all_markets_list).drop_duplicates(subset='ticker') if all_markets_list else pd.DataFrame()
print(f'\nTotal unique markets: {len(all_markets)}')

# Pre-filter: drop short-duration bracket markets (open < 7 days)
# and markets with zero open interest in the catalog
if 'close_time' in all_markets.columns and 'open_time' in all_markets.columns:
    all_markets['open_time_dt']  = pd.to_datetime(all_markets['open_time'],  errors='coerce', utc=True)
    all_markets['close_time_dt'] = pd.to_datetime(all_markets['close_time'], errors='coerce', utc=True)
    duration_days = (all_markets['close_time_dt'] - all_markets['open_time_dt']).dt.days
    all_markets = all_markets[duration_days >= 7]
    print(f'After duration filter (>=7 days): {len(all_markets)} markets')

print(f'\nSample tickers:\n{all_markets["ticker"].head(20).tolist()}')

Fetching event markets from targeted series...
  KXBTCMAXY: 19 markets
  KXBTCMAX100: 8 markets
  KXBTCMAX125: 0 markets
  KXBTCMAX150: 10 markets
  KXBTC2025100: 1 markets
  KXBTCRESERVE: 2 markets
  KXUSDCMIN: 1 markets
  KXCRYPTO: 0 markets
  KXSOLANAETF: 0 markets
  KXFED: 142 markets
  KXCPI: 99 markets
  KXGDP: 36 markets
  KXUNEMPLOYMENT: 0 markets

Total unique markets: 318
After duration filter (>=7 days): 316 markets

Sample tickers:
['KXBTCMAXY-26DEC31-109999.99', 'KXBTCMAXY-26DEC31-99999.99', 'KXBTCMAXY-26DEC31-149999.99', 'KXBTCMAXY-26DEC31-139999.99', 'KXBTCMAXY-26DEC31-129999.99', 'KXBTCMAXY-26DEC31-119999.99', 'KXBTCMAXY-26DEC31-199999.99', 'KXBTCMAXY-25-DEC31-224999.99', 'KXBTCMAXY-25-DEC31-189999.99', 'KXBTCMAXY-25-DEC31-169999.99', 'KXBTCMAXY-25-DEC31-139999.99', 'KXBTCMAXY-25-DEC31-129999.99', 'KXBTCMAXY-25-DEC31-499999.99', 'KXBTCMAXY-25-DEC31-299999.99', 'KXBTCMAXY-25-DEC31-249999.99', 'KXBTCMAXY-25-DEC31-199999.99', 'KXBTCMAXY-25-DEC31-179999.99', 'KXBTCMAXY-25-D

## 2. Classify by Tier

- **Tier 1**: Direct crypto (BTC/ETH/SOL price, ETF approval, Bitcoin reserve, USDC de-peg)
- **Tier 2**: Regulatory/institutional (stablecoin bills, SEC, MicroStrategy, Coinbase)
- **Tier 3**: Macro (FOMC, CPI, inflation, unemployment)
- **Tier 4**: Exclude (sports, entertainment, weather)

In [3]:
def classify_tier(row, cfg):
    text = ' '.join([
        str(row.get('title', '')),
        str(row.get('ticker', '')),
        str(row.get('event_ticker', '')),
        str(row.get('yes_sub_title', '')),
        str(row.get('no_sub_title', '')),
    ]).lower()

    tier_kws = cfg['market_tiers']

    if any(kw.lower() in text for kw in tier_kws['tier4_exclude_keywords']):
        return 4
    if any(kw.lower() in text for kw in tier_kws['tier1_keywords']):
        return 1
    if any(kw.lower() in text for kw in tier_kws['tier2_keywords']):
        return 2
    if any(kw.lower() in text for kw in tier_kws['tier3_keywords']):
        return 3
    return 0  # unclassified

all_markets['tier'] = all_markets.apply(classify_tier, axis=1, cfg=cfg)

tier_counts = all_markets['tier'].value_counts().sort_index()
print('Markets per tier:')
for t, n in tier_counts.items():
    label = {0: 'Unclassified', 1: 'Tier1 Direct Crypto', 2: 'Tier2 Regulatory',
             3: 'Tier3 Macro', 4: 'Tier4 Exclude'}
    print(f'  Tier {t} ({label.get(t, "")}): {n}')

Markets per tier:
  Tier 1 (Tier1 Direct Crypto): 39
  Tier 3 (Tier3 Macro): 277


In [4]:
# Inspect Tier 1 and 2 markets
relevant = all_markets[all_markets['tier'].isin([1, 2, 3])].copy()

display_cols = ['ticker', 'tier', 'title', 'status', 'volume_24h_fp', 'open_interest_fp']
display_cols = [c for c in display_cols if c in relevant.columns]
relevant[display_cols].sort_values(['tier', 'ticker']).to_string(max_rows=100)

"                               ticker  tier                                                                                                    title       status volume_24h_fp open_interest_fp\n26                 KXBTCMAX100-26-APR     1                                              Will Bitcoin be above $100000 by May 1, 2026 at 12:00AM ET?       active      12702.00        490245.00\n19                 KXBTCMAX100-26-DEC     1                                           Will Bitcoin be above $100000.00 by Jan 1, 2027 at 12:00AM ET?       active       1647.00         65538.00\n25                 KXBTCMAX100-26-FEB     1                                          Will Bitcoin be above $100000 by March 1st, 2026 at 12:00AM ET?    finalized          0.00        370511.00\n24                 KXBTCMAX100-26-JAN     1                                         Will Bitcoin be above $100000 by February 1, 2026 at 12:00AM ET?    finalized          0.00        143904.00\n23                KXBTCMAX100

## 3. Apply Liquidity Filter

For each candidate market, fetch its candlestick data and check:
- Average daily volume >= $5,000
- Mean bid-ask spread <= 8%
- Peak open interest >= $25,000
- Active for >= 30 days

In [ ]:
import time
from tqdm.notebook import tqdm

# Re-derive 'relevant' if this cell is run before cell-6
if 'relevant' not in dir() or len(relevant) == 0:
    if 'all_markets' not in dir() or len(all_markets) == 0:
        raise RuntimeError("'all_markets' not defined. Run cells 1–5 first.")
    relevant = all_markets[all_markets['tier'].isin([1, 2, 3])].copy()
    print(f"Re-derived relevant: {len(relevant)} markets")

liq_cfg = cfg['liquidity_filter']
liquidity_results = []

candidate_tickers = relevant['ticker'].dropna().unique().tolist()
print(f'Checking liquidity for {len(candidate_tickers)} markets...')
print(f'  min_days={liq_cfg["min_active_days"]}  '
      f'min_vol={liq_cfg["min_daily_volume_usd"]:,} contracts  '
      f'max_spread={liq_cfg["max_bid_ask_spread_pct"]:.0%}  '
      f'min_oi={liq_cfg["min_open_interest_usd"]:,} contracts')
print()

def fetch_with_retry(ticker, is_hist, max_retries=3):
    """Fetch candlesticks with exponential backoff on 429."""
    for attempt in range(max_retries):
        try:
            df = fetch_kalshi_candlesticks(
                ticker=ticker,
                start_ts=int(pd.Timestamp('2022-01-01', tz='UTC').timestamp()),
                end_ts=int(pd.Timestamp.now(tz='UTC').timestamp()),
                period_interval=1440,
                is_historical=is_hist,
                api_key=API_KEY,
            )
            return df, None
        except Exception as e:
            err = str(e)
            if '429' in err and attempt < max_retries - 1:
                wait = 2 ** (attempt + 1)  # 2, 4 seconds
                time.sleep(wait)
            else:
                return None, err
    return None, 'max retries exceeded'

for ticker in tqdm(candidate_tickers):
    row = relevant[relevant['ticker'] == ticker].iloc[0]
    is_hist = str(row.get('status', '')).lower() in ('settled', 'closed', 'finalized')
    time.sleep(0.15)  # 150ms between calls (~6 req/s, well under rate limit)

    df, err = fetch_with_retry(ticker, is_hist)

    if err:
        print(f'  {ticker} [T{row.get("tier","?")}]: ERROR — {err[:80]}')
        liquidity_results.append({'ticker': ticker, 'tier': row.get('tier', 0),
                                   'passes': False, 'reason': err[:60]})
        continue

    if df is None or df.empty:
        liquidity_results.append({'ticker': ticker, 'tier': row['tier'],
                                   'passes': False, 'reason': 'no data'})
        continue

    active_days   = len(df)
    avg_daily_vol = pd.to_numeric(df.get('volume',        pd.Series([0])), errors='coerce').mean()
    mean_spread   = pd.to_numeric(df.get('spread_pct',    pd.Series([np.nan])), errors='coerce').mean()
    max_oi        = pd.to_numeric(df.get('open_interest', pd.Series([0])), errors='coerce').max()

    # Spread: NaN or <= threshold (threshold is 1.0 = disabled)
    spread_ok = pd.isna(mean_spread) or (mean_spread <= liq_cfg['max_bid_ask_spread_pct'])

    passes = (
        active_days   >= liq_cfg['min_active_days'] and
        avg_daily_vol >= liq_cfg['min_daily_volume_usd'] and
        spread_ok and
        max_oi        >= liq_cfg['min_open_interest_usd']
    )

    status = 'PASS ✓' if passes else 'FAIL ✗'
    reasons = []
    if active_days   < liq_cfg['min_active_days']:       reasons.append(f'days={active_days}<{liq_cfg["min_active_days"]}')
    if avg_daily_vol < liq_cfg['min_daily_volume_usd']:  reasons.append(f'vol={avg_daily_vol:,.0f}<{liq_cfg["min_daily_volume_usd"]:,}')
    if not spread_ok:                                    reasons.append(f'spread={mean_spread:.0%}>{liq_cfg["max_bid_ask_spread_pct"]:.0%}')
    if max_oi        < liq_cfg['min_open_interest_usd']: reasons.append(f'oi={max_oi:,.0f}<{liq_cfg["min_open_interest_usd"]:,}')
    fail_str = ' | '.join(reasons)
    spread_str = f'{mean_spread:.0%}' if not pd.isna(mean_spread) else 'n/a'
    print(f'  {ticker} [T{row["tier"]}] {status}  '
          f'days={active_days:>3}  vol={avg_daily_vol:>8,.0f}  spread={spread_str:>5}  oi={max_oi:>10,.0f}'
          + (f'  ({fail_str})' if fail_str else ''))

    liquidity_results.append({
        'ticker': ticker, 'tier': row['tier'],
        'active_days': active_days,
        'avg_daily_vol_usd': round(avg_daily_vol, 0),
        'mean_spread_pct': round(mean_spread, 4) if not pd.isna(mean_spread) else np.nan,
        'max_oi_usd': round(max_oi, 0),
        'passes': passes,
        'reason': 'ok' if passes else fail_str,
    })

liq_df = pd.DataFrame(liquidity_results)
n_pass = int(liq_df['passes'].sum())
print(f'\n{"="*60}')
print(f'Passes: {n_pass} / {len(liq_df)}')
if n_pass > 0:
    print('\nPassing markets:')
    passing = liq_df[liq_df['passes']].sort_values(['tier', 'avg_daily_vol_usd'], ascending=[True, False])
    print(passing[['ticker', 'tier', 'active_days', 'avg_daily_vol_usd', 'mean_spread_pct', 'max_oi_usd']].to_string(index=False))
else:
    print('\nWARNING: Zero markets. Fail reasons:')
    print(liq_df['reason'].value_counts().head(10).to_string())

In [ ]:
out_dir = Path('../data/raw/kalshi')
out_dir.mkdir(parents=True, exist_ok=True)

final_markets = liq_df[liq_df['passes']].sort_values(['tier', 'avg_daily_vol_usd'], ascending=[True, False])

all_markets.to_parquet(out_dir / 'market_catalog.parquet', index=False)
final_markets.to_parquet(out_dir / 'final_market_list.parquet', index=False)
liq_df.to_parquet(out_dir / 'liquidity_screen.parquet', index=False)
final_markets.to_csv(out_dir / 'final_market_list.csv', index=False)

print(f'Saved market_catalog.parquet     : {len(all_markets)} markets')
print(f'Saved liquidity_screen.parquet   : {len(liq_df)} markets evaluated')
print(f'Saved final_market_list.parquet  : {len(final_markets)} markets passed')
print()
print('FINAL MARKET LIST:')
print(final_markets[['ticker', 'tier', 'active_days', 'avg_daily_vol_usd', 'max_oi_usd']].to_string(index=False))